# Project.py

In [1]:
import logging
logging.basicConfig(
    level=logging.INFO,
    force=True,
)

import warnings
warnings.filterwarnings(action='ignore')

from polymerist.rdutils import disable_kekulized_drawing
disable_kekulized_drawing()

INFO:rdkit:Enabling RDKit 2024.09.5 jupyter extensions
INFO:numexpr.utils:Note: NumExpr detected 64 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [2]:
from pathlib import Path
from src.project import (
    PolymerBuildProject,
    load_job_rdmol,
    atoms_validated,
    mechanism_established,
)

# project_path = Path('polyID_test')
project_path = Path('polyID_production')
project = PolymerBuildProject.get_project(project_path)
print(len(project))

[19:06:21] WARNING: not removing hydrogen atom with dummy atom neighbors
INFO:polymerist.smileslib.functgroups:Loading functional group SMARTS data from LUT
INFO:src.reactions:Initializing reaction template 1/8 ("polyester")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:Initializing reaction template 2/8 ("polyamide")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:Initializing reaction template 3/8 ("polyimide")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:Initializing reaction template 4/8 ("polycarbonate_phosgene")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:Initializing reaction template 5/8 ("polycarbonate_nonphosgene")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:Initializing reaction template 6/8 ("polyurethane_isocyanate")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:I

5056


### Run Jobs

In [ ]:
project.run(
    names=[
        # 'polymerize',
        # 'oligomerize',
        # 'pack_lattice',
        # 'to_interchange',
        'md_export'
    ],
    # jobs=[
    #     project.open_job(id='214e8512b6218c21668ce8c19c6bf359')
    # ]
)

## Mapping results between prior and new results

In [28]:
import pandas as pd

old_project_path = Path('polyID_production_LAMMPS')
old_project = PolymerBuildProject.get_project(old_project_path)
# print(len(old_project))

old_id_map = {(job.doc.smiles_original, job.sp.DOP) : job.id for job in old_project}
new_id_map = {(job.doc.smiles_original, job.sp.DOP) : job.id for job in project}

compatibility_map = {}
num_unmatched : int = 0
for statekey, old_id in old_map.items():
    new_id = new_map.get(statekey)
    compatibility_map[old_id] = new_id
    if new_id is None:
        num_unmatched += 1
print(num_unmatched)

compat_df = pd.DataFrame(compatibility_map.items(), columns=['Pilot set hash', 'Expanded set hash'])
compat_df.to_csv('pilot_to_prod_hashes.csv')

90


## Re-marking sulfur-containing chemistries as eligible for build

In [ ]:
from rich.progress import track
from rdkit.Chem.rdqueries import AtomNumEqualsQueryAtom

sulfur_query = AtomNumEqualsQueryAtom(16, negate=False)

sulfur_hashes : set[str] = set()
for job in track(project, description='Looking for sulfur atoms', total=len(project)):
    mol = load_job_rdmol(job, separate_mols=False)
    if any(mol.GetAtomsMatchingQuery(sulfur_query)):
        sulfur_hashes.add(job.id)

print(len(sulfur_hashes))

In [ ]:
from collections import Counter

c = Counter()
for job_hash in sulfur_hashes:
    job = project.open_job(id=job_hash)
    job.doc.pop('has_banned_atom_types')
    c[atoms_validated(job)] += 1 # verify that this no longer marks these as validated

print(c)

## Checking that aromaticity is being respected

In [ ]:
from src.utils.dataIO import read_rxn_mapping_data
from polymerist.rdutils.reactions.reactions import AnnotatedReaction


RXN_DIR : Path = Path('src/reactions')
# RXN_PATHNAME : str = 'rxn_smarts.json'
RXN_PATHNAME : str = 'rxns_polyID.json'

show : bool = not True

# load reactions
rxns : dict[str, AnnotatedReaction] = {}
for rxnname, smarts in read_rxn_mapping_data(RXN_DIR / RXN_PATHNAME).items():
    rxn = AnnotatedReaction.from_smarts(smarts)
    rxns[rxnname] = rxn 
    if show:
        print(rxnname)
        display(rxn)

In [ ]:
from rdkit import Chem
from rdkit.Chem.rdmolops import SANITIZE_ALL, AROMATICITY_MDL
from polymerist.rdutils.sanitization import sanitize_mol

from polymerist.rdutils.reactions.reactors import PolymerizationReactor
from polymerist.rdutils.reactions.fragment import CutMinimumCostBondsStrategy

build_jobs = [job for job in project if mechanism_established(job)]
job = build_jobs[0]
job = project.open_job(id='455fc6cfdb0fc1e7e8ce8c61ce512567')

monomers = PolymerBuildProject.sanitized_mol_from_smiles(job.sp.smiles_explicit, separate_mols=True)
for m in monomers:
    display(m)

In [ ]:
job = project.open_job(id='57825dbdb6cab389c8f39ad2db3b4c1d')

In [ ]:
import pandas as pd

records : list[dict] = []
for job in project:
    op_times = job.doc.get(PolymerBuildProject.OP_TIME_RECORD_NAME)
    if op_times is not None:
        op_times['Job ID'] = job.id
        records.append(op_times)